<a href="https://colab.research.google.com/github/alfredo170987/Proyecto-1-Sistemas-de-Comunicaci-n/blob/main/Notebook%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 1: Autoencoders para Comunicaciones de Extremo a Extremo (Base)
El objetivo de este notebook es replicar el paper clásico de O'Shea ("An Introduction to Deep Learning for the Physical Layer"), que demostró que una red neuronal puede aprender sistemas de modulación (como BPSK o QAM) de forma autónoma.

• Entrada: Vectores de bits aleatorios (representados como vectores one-hot).

• Encoder: Capas densas (nn.Linear) que comprimen los bits en un espacio bidimensional (I/Q: En fase y cuadratura).

• Restricción de potencia: Aplicar torch.mean(x ** 2) para normalizar la constelación aprendida.

• Canal: Capa personalizada que añade ruido AWGN usando tu línea torch.randn_like.

• Decoder: Capas densas que intentan clasificar qué bits se enviaron originalmente.

• Métricas a graficar:

	• La constelación aprendida en el plano 2D (verás cómo la red agrupa los puntos de forma similar a una modulación tradicional para protegerse del ruido).
	• La curva BER vs. SNR (Tasa de error de bit frente a la relación señal-ruido).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# 1. HIPERPARÁMETROS DEL SISTEMA
M = 16          # Número de mensajes posibles (ej. equivalente a una modulación 16-QAM)
k = int(np.log2(M)) # Bits por mensaje (4 bits)
dim_canal = 2   # Dimensión del canal (2 canales: I y Q, En Fase y Cuadratura)

print(f"Sistemas de transmisión de {k} bits ({M} mensajes únicos) en un espacio de {dim_canal}D.")

# 2. DEFINICIÓN DEL AUTOENCODER DE CANAL
class AutoencoderComunicaciones(nn.Module):
    def __init__(self, M, dim_canal):
        super(AutoencoderComunicaciones, self).__init__()

        # El Transmisor (Encoder) mapea el mensaje 'one-hot' al espacio del canal
        self.encoder = nn.Sequential(
            nn.Linear(M, M),
            nn.ReLU(),
            nn.Linear(M, dim_canal) # Salida: coordenadas I y Q
        )

        # El Receptor (Decoder) toma los símbolos ruidosos y clasifica el mensaje original
        self.decoder = nn.Sequential(
            nn.Linear(dim_canal, M),
            nn.ReLU(),
            nn.Linear(M, M) # Salida: Logits para clasificación (CrossEntropy)
        )

    def forward(self, x, snr_dB):
        # --- TRANSMISOR (ENCODER) ---
        x_símbolos = self.encoder(x)

        # RESTRECCIÓN DE POTENCIA (Tus fórmulas)
        # Aseguramos que la potencia promedio de los símbolos sea exactamente 1
        potencia = torch.mean(x_símbolos ** 2)
        x_tx = x_símbolos / torch.sqrt(potencia)

        # --- CANAL AWGN (Tus fórmulas) ---
        snr_lineal = 10 ** (snr_dB / 10.0)
        # Como dividimos la potencia entre dim_canal, ajustamos la varianza del ruido
        noise_var = 1.0 / (2 * dim_canal * snr_lineal)

        ruido = torch.randn_like(x_tx) * torch.sqrt(torch.tensor(noise_var))
        x_rx = x_tx + ruido

        # --- RECEPTOR (DECODER) ---
        out = self.decoder(x_rx)
        return out, x_tx

# 3. CONFIGURACIÓN DEL ENTRENAMIENTO
modelo = AutoencoderComunicaciones(M, dim_canal)
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.005)

# Generación de datos de entrenamiento (Mensajes aleatorios del 0 al M-1)
N_ENTRENAMIENTO = 50000
mensajes_entrenamiento = torch.randint(0, M, (N_ENTRENAMIENTO,))
# Convertir a vectores One-Hot para que entren a la red neuronal
x_entrenamiento = nn.functional.one_hot(mensajes_entrenamiento, num_classes=M).float()

# SNR fija para el entrenamiento (usualmente un valor medio, ej. 7 dB)
snr_entrenamiento = 7.0

# 4. BUCLE DE ENTRENAMIENTO
ÉPOCAS = 20
tam_lote = 1024

print("Iniciando entrenamiento del Autoencoder...")
for epoca in range(ÉPOCAS):
    pérdida_total = 0
    # Mezclar datos manualmente por lotes
    permutacion = torch.randperm(x_entrenamiento.size(0))

    for i in range(0, x_entrenamiento.size(0), tam_lote):
        indices = permutacion[i:i+tam_lote]
        lote_x = x_entrenamiento[indices]
        lote_y = mensajes_entrenamiento[indices]

        optimizador.zero_grad()
        predicciones, _ = modelo(lote_x, snr_entrenamiento)

        loss = criterio(predicciones, lote_y)
        loss.backward()
        optimizador.step()

        pérdida_total += loss.item()

    print(f"Época {epoca+1}/{ÉPOCAS} - Pérdida: {pérdida_total/(N_ENTRENAMIENTO/tam_lote):.4f}")

Notebook 2: Deep JSCC para Transmisión de Imágenes sobre Canales Inalámbricos

Aquí subes el nivel aplicando Codificación Conjunta de Fuente y Canal (JSCC) real. En lugar de transmitir bits, el Autoencoder aprende a transmitir características semánticas de una imagen directamente como símbolos analógicos a través del canal.

• Entrada: Dataset de imágenes pequeñas (como CIFAR-10 o MNIST).

• Encoder: Capas Convolucionales (nn.Conv2d) que reducen la dimensionalidad de la imagen (compresión de fuente) y la preparan para el canal (compresión de canal).

• Canal: Simulación de un canal inalámbrico variable (puedes empezar con AWGN y, si quieres un extra, simular un canal con desvanecimiento tipo Rayleigh).
• Decoder: Capas Convolucionales Transpuestas o Upsampling (nn.ConvTranspose2d) que reconstruyen la imagen a partir de los símbolos ruidosos recibidos.

• Función de Pérdida: Error Cuadrático Medio (MSE) entre la imagen original y la reconstruida.

• Métricas a graficar:

	• Visualización de imágenes originales vs. reconstruidas a diferentes niveles de ruido (baja SNR vs. alta SNR).
	• Curva de PSNR (Peak Signal-to-Noise Ratio) o SSIM vs. SNR del canal para demostrar el efecto de "degradación suave" (a diferencia de los sistemas tradicionales, JSCC no sufre el efecto acantilado donde la comunicación se corta de golpe al bajar la señal).